# Straight-Through Estimator (STE)

Quantization-Aware Training (QAT) simulates the effects of quantization during the training or fine-tuning process. This simulation happens by inserting operations, often called "fake quantization" nodes, into the model's computational graph

These nodes take high-precision inputs (like FP32 activations or weights) and produce low-precision outputs (simulating INT8 or INT4, for example), which are then used in the subsequent operations.

However, this introduces a significant challenge for training algorithms that rely on gradient descent. 

The core operation within fake quantization is the rounding function, which maps a continuous range of inputs to a discrete set of output values. Mathematically, this function is a step function.

Consider a simple rounding function, Round(x). Its derivative is zero almost everywhere (on the flat steps) and undefined at the points where the value jumps. Standard backpropagation relies on calculating gradients to update model weights. If the gradient is zero or undefined, the updates cannot flow back through the quantization operation to the preceding layers and weights. This effectively stalls the learning process for parameters situated before the quantization node.

- "Vì phép làm tròn có đạo hàm bằng 0 ở hầu hết mọi điểm và không xác định tại điểm nhảy, nên khi dùng backprop, gradient không thể truyền qua phép quantization. Do đó, các tham số ở phía trước quantization không được cập nhật, khiến quá trình học bị tắc nghẽn."

How can the model learn to adapt to quantization if the gradients are blocked? This is where the $\text{Straight-Through Estimator (STE)}$ comes into play.

## The problem: Non-Differentiable Quantization

Let's represent the quantization function (including scaling, rounding, and de-quantization back to a float that mimics the low-precision step) as $y = q(x)$.

In the forward pass of training, we compute $y$ from the input $x$ and use $y$ in subsequent calculations.

During the backward pass, we need to compute the gradient of the loss $L$ with respect to the input $x$, denoted as $\frac{\partial L}{\partial x}$. Using the chain rule, this would normally be calculated as:
$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y}.\frac{\partial y}{\partial x}
$$

The problem lies in the term $\frac{\partial y}{\partial x} = \frac{\partial q(x)}{\partial x}$. 

As mentioned, this derivative is problematic (mostly 0, undefined at jumps).

If we use this true gradient, $\frac{\partial L}{\partial x}$ becomes zero almost everywhere, preventing weight updates.

## The solution: Straight-Through Estimator (STE)

The Straight-Through Estimator (STE) provides a practical workaround for this issue. It's an approximation used specifically during the backward pass. The core idea is simple:

1. $\text{Forward Pass}$: Compute the quantization operation as usual: $y = q(x)$. The effects of quantization (precision loss) are fully applied.

2. $\text{Backward Pass}$: When computing the gradient $\frac{\partial L}{\partial x}$, ignore the true derivative of $q(x)$ and instead $approximate$ it. The most common STE approximation sets the gradient $\frac{\partial y}{\partial x}$ to $1$.

Therefore, during backpropagation with STE, the gradient calculation becomes:

$$
\frac{\partial L}{\partial x} ≈ \frac{\partial L}{\partial y}.1 = \frac{\partial L}{\partial y}
$$

This means the gradient $\frac{\partial L}{\partial y}$ computed for the output of the quantization node is passed "straight through" to become the gradient $\frac{\partial L}{\partial x}$ for the input, as if the quantization operation was an identity function $(y = x)$ only for the gradient calculation.

![](image1.png)

The diagram illustrates the STE process:
- The forward pass applies the actual quantization $q(x)$. 
- The backward pass uses the STE approximation $(\frac{\partial y}{\partial x} ≈ 1)$ to allow the incoming gradient $\frac{\partial L}{\partial y}$ to pass through unchanged to become $\frac{\partial L}{\partial x}$.

## Why does STE Work?

It might seem counter-intuitive to use an approximation that ignores the true nature of the quantization function during backpropagation. However, STE works well in practice for several reasons:

- $\text{Gradient Flow}$: It ensures that gradients are not blocked, allowing optimization signals to reach the weights preceding the quantization node.

- $\text{Learning Adaptation}$: Although the gradient calculation is an approximation, the forward pass still uses the actual quantized values. The loss function reflects the errors introduced by this quantization. By allowing gradients to flow, STE enables the model weights to adjust in a way that minimizes this quantization-induced error. The network learns parameters that are more resilient to the rounding effects applied during the forward pass.

- $\text{Simplicity}$: STE is easy to implement within standard deep learning frameworks. Most frameworks providing QAT capabilities incorporate STE internally.

## Variants of STE

While the identity approximation ($\frac{\partial y}{\partial x} = 1$) is the most common form of STE, some variations exist.

For instance, another approach involves clipping the gradient based on the input range used for quantization. If the quantization function effectively clips inputs $x$ to a range $[c_{min}, c_{max}]$ before quantizing, the STE might be defined as:

$$
\frac{\partial y}{\partial x} =
\begin{cases}
1 & if \space c_{min} \leq x \leq c_{max} \\
0 & otherwise
\end{cases}
$$

This variant prevents gradients from flowing back for inputs that were already outside the quantization range, which can sometimes help stabilize training. However, the simple identity approximation often suffices.

In summary, the Straight-Through Estimator is a fundamental technique that makes Quantization-Aware Training possible. By providing a path for gradients through the non-differentiable quantization operations, it allows deep learning models to adapt their weights during training or fine-tuning, leading to significantly better accuracy for quantized models compared to Post-Training Quantization, especially at very low bit-widths.